# Introduction to the Deep Generative Decoder (DGD)

A short, practical introduction to the **DGD** and to this repository (**ImageDGD**: a DGD trained on FashionMNIST).
After this notebook you should be able to

1. explain what a DGD is and how it differs from an autoencoder / VAE,
2. follow how a DGD is trained and how it represents new images (in formulas and in words),
3. find your way around the code, the `config.yaml` and its hyperparameters,
4. train a model, and read the figures a run produces.

**How to use it:** read top to bottom. The few code cells are quick; the heavy parts (a full training run) live in
`dgd_training_demo.ipynb`. All figures below come from one real 200-epoch run with the default `config/config.yaml`
(copied to `notebooks/assets/intro/` together with a few short videos, so everything is visible without re-running anything).

**Background reading** (all by the Krogh group, full references at the end):
*Schuster & Krogh 2021* (the idea, from a manifold-learning view) and *Schuster & Krogh 2023* (the DGD itself).

## 1. What is a DGD?

An **autoencoder** has two networks: an *encoder* that maps an image $x$ to a low-dimensional code $z$, and a *decoder*
that maps $z$ back to an image. A **VAE** additionally pushes every code towards a fixed Gaussian.

The **Deep Generative Decoder** keeps only the decoder. There is **no encoder**. Instead, every training image $x_i$
gets its own free vector $z_i$ (its *representation*), stored as one row of a big table, and $z_i$ is learned by gradient
descent in exactly the same way as the network weights. A learnable **Gaussian mixture model (GMM)** acts as the prior
over the representations, so similar images end up in the same mixture component.

| | Autoencoder / VAE | DGD |
|---|---|---|
| Encoder network | yes | **no** |
| Code of a training image | output of the encoder | a **free parameter** $z_i$ (a row in a table) |
| Prior over codes | none / fixed $\mathcal N(0, I)$ | **learnable GMM** $p(z\mid\varphi)$ |
| Training objective | reconstruction (+ KL / ELBO) | **MAP**: reconstruction + log-prior |
| Code of a *new* image | one forward pass through the encoder | optimise $z$ for that image with the decoder frozen (*inference*) |
| Sampling new images | sample $z$ from the prior, decode | sample $z$ from the GMM, decode |

### The objective

The DGD finds the **maximum a posteriori (MAP)** estimate of everything at once. With $\theta$ the decoder weights,
$\varphi$ the GMM parameters and $Z=\{z_1,\dots,z_N\}$ the representations (Schuster & Krogh 2023, eq. 3):

$$
\arg\max_{Z,\varphi,\theta}\;\; \sum_{i=1}^{N}\Big[\log P(x_i\mid z_i,\theta) + \log P(z_i\mid\varphi)\Big] + \log P(\varphi) + \log P(\theta)
$$

In this repository that becomes the loss we minimise (per batch):

$$
\mathcal L \;=\; \underbrace{\sum_i \big\|f_\theta(\tilde z_i)-x_i\big\|_2^2}_{\text{reconstruction}\;\leftrightarrow\;\log P(x\mid z,\theta)}
\;-\;\lambda\underbrace{\sum_i \log p_{\text{GMM}}(\tilde z_i)}_{\text{prior}\;\leftrightarrow\;\log P(z\mid\varphi)},
\qquad \tilde z_i = z_i+\epsilon,\;\; \epsilon\sim\mathcal N\big(0,\tfrac{\sigma_t^2}{d}I\big)
$$

* the squared error is a Gaussian likelihood with fixed variance,
* $\log P(\theta)$ is the weight decay of the decoder optimiser (a Gaussian prior on the weights),
* $\lambda$ is `training.lambda_gmm`,
* $\epsilon$ is *latent noise injection*, a regulariser whose strength $\sigma_t$ is annealed during training (see `noise_injection_explained.ipynb`).

### Why does dropping the encoder work?

*Schuster & Krogh 2021* view the decoder as defining a low-dimensional **manifold** in image space. The best
representation of an image is the point in $z$-space whose decoded image is closest to it, and training with a free $z_i$
per image finds exactly that. Their main points:

* an encoder only adds *constraints* on the representations, so with the same decoder it cannot improve the fit to the training data,
* counting parameters vs. constraints, the decoder is far better determined than an encoder by the same number of samples (by up to about $n/m$ times, with $n$ the image size and $m$ the latent size). So a decoder trained alone generalises better from **small** datasets,
* the price: to represent a new image you must run a small optimisation instead of a single forward pass.

## 2. The architecture

A DGD has **three groups of learnable things**, each updated differently:

| Parameter group | What it is | Code | Updated |
|---|---|---|---|
| Representations $Z$ | table of $N\times d$ numbers, one row per image | `RepresentationLayer` | AdamW, **once per epoch** (gradients accumulate over all batches) |
| Decoder $\theta$ | convolutional network $z\mapsto\hat x$ | `ConvDecoder` | AdamW, **every batch** |
| GMM $\varphi$ | $K$ Gaussian components over $z$-space | `tgmm.GaussianMixture` | **EM refit** on the current $Z$ every few epochs (not by gradient descent) |

### The decoder

The decoder is a function $f_\theta:\mathbb R^{d}\to[0,1]^{1\times 28\times 28}$ from a latent vector to an image. In the default configuration ($d=4$):

$$
z\;\xrightarrow{\ \text{Linear}\ }\;\mathbb R^{3136}\;\xrightarrow{\ \text{reshape}\ }\;64{\times}7{\times}7
\;\xrightarrow{\ \text{block}\ }\;32{\times}14{\times}14
\;\xrightarrow{\ \text{block}\ }\;32{\times}28{\times}28
\;\xrightarrow{\ \text{conv},\ \sigma\ }\;\hat x\in[0,1]^{1\times28\times28}
$$

where each *block* is: nearest-neighbour upsampling ($\times2$), $3\times3$ convolution, BatchNorm, LeakyReLU (BatchNorm and LeakyReLU are also applied right after the reshape). The final convolution produces one channel and the sigmoid $\sigma$ keeps pixel values in $[0,1]$, matching the data.
The decoder has only 43,937 parameters; its capacity is set by `model.decoder.hidden_dims` (here `[64, 32]`).

The decoder is the only part that is *image specific*; everything else (representations, GMM, training loop) would stay
the same for another data type. The next cell builds it from the config so you can check the shapes yourself.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import torch
from omegaconf import OmegaConf
from hydra import initialize, compose

# make `import src...` work whether the notebook is run from the repo root or from notebooks/
project_root = Path.cwd().parent if "notebooks" in Path.cwd().parts else Path.cwd()
sys.path.append(str(project_root))

from src.models import ConvDecoder, RepresentationLayer

with initialize(version_base=None, config_path="../config"):
    config = compose(config_name="config")

dc = config.model.decoder
decoder = ConvDecoder(
    latent_dim=config.model.representation.n_features,
    hidden_dims=list(dc.hidden_dims), output_channels=dc.output_channels,
    output_size=tuple(dc.output_size), init_size=tuple(dc.init_size),
    kernel_size=dc.kernel_size, stride=dc.stride, padding=dc.padding, output_padding=dc.output_padding,
    normalization=dc.normalization, activation=dc.activation, final_activation=dc.final_activation,
    dropout_rate=dc.dropout_rate, upsampling_mode=dc.upsampling_mode,
)
print(decoder)

z = torch.zeros(5, config.model.representation.n_features)   # 5 latent vectors
print("\nlatent batch:", tuple(z.shape), "-> decoded images:", tuple(decoder(z).shape))

# The 'encoder-free' trade-off in numbers: the table of representations is *larger* than the decoder.
n_train = int(70_000 * (1 - config.data.val_split - config.data.test_split))
n_dec = sum(p.numel() for p in decoder.parameters())
n_rep = n_train * config.model.representation.n_features
print(f"\ndecoder parameters        : {n_dec:>9,}")
print(f"representation parameters : {n_rep:>9,}  ({n_train:,} training images x d={config.model.representation.n_features})")

## 3. How training and inference work

### Training

`DGDTrainer.train` in `src/training/trainer.py` (compare Algorithm 1 in Schuster & Krogh 2023). Initially every row $z_i$ of $Z$ (and of the validation table $Z_{\text{val}}$) is zero, the decoder weights are random and the GMM is not fitted yet. Then, for each epoch:

1. **Noise level.** The noise strength $\sigma_t$ is taken from the schedule (cosine from 1.0 down to 0.01). It is the expected size of the displacement in $z$-space, so the per-coordinate standard deviation is $\sigma_t/\sqrt d$.
2. **GMM fit.** From epoch `first_epoch_gmm` = 50 on, the GMM is fitted by EM to the current (noised) representations $Z+\epsilon$: a full refit every `refit_gmm_interval` = 50 epochs and a cheap warm-started update on the epochs in between. Before that epoch there is no prior.
3. **Training phase.** For each batch of images $x_i$ take their rows $z_i$, add noise, decode and compute

   $$
   \mathcal L=\sum_{i\in\text{batch}}\big\|f_\theta(\tilde z_i)-x_i\big\|_2^2\;-\;\lambda\,\mathbb 1[\text{epoch}\ge\text{first\_epoch\_gmm}]\sum_{i\in\text{batch}}\log p_{\text{GMM}}(\tilde z_i),\qquad \tilde z_i=z_i+\epsilon_i,\ \ \epsilon_i\sim\mathcal N(0,\tfrac{\sigma_t^2}{d}I).
   $$

   One backward pass gives the gradient with respect to the decoder weights $\theta$ **and** with respect to the rows $z_i$. The decoder takes an optimiser step right away (**every batch**). The gradients of the rows accumulate, and since every image appears in exactly one batch per epoch, $Z$ takes **one step per epoch**. The noise is only used in the forward pass; the stored $z_i$ never contains it.
4. **Validation phase.** The decoder is frozen and the same loss is computed for the validation images. Only $Z_{\text{val}}$ receives gradients (one step per epoch). So the validation loss measures how well the model can represent images it was never trained on: this is exactly the inference procedure below, run every epoch.
5. **Model selection.** Keep the epoch with the lowest validation *reconstruction* loss. (The total loss is not a good criterion: the GMM term can keep decreasing as the mixture covariances shrink, even when reconstructions get worse.) At the end, restore that epoch and refit the GMM on its representations.

### Inference

For new images the decoder $\theta$ and the GMM $\varphi$ are frozen and only the new representations are optimised, for $M$ = `training.inference.epochs` = 400 steps starting from $z=0$:

$$
z_i^{*}=\arg\min_{z}\;\big\|f_\theta(z+\epsilon_m)-x_i\big\|_2^2\;-\;\lambda\,\mathbb 1[m>M_0]\,\log p_{\text{GMM}}(z+\epsilon_m),
$$

with the same kind of annealed noise $\epsilon_m$, and the prior switched on only after $M_0$ = `prior_warmup_steps` = 50 steps (Algorithm 2 in the paper; run by `dgd_test_inference.ipynb`, explained with results in section 8).

### What differs from the paper

The paper updates the GMM by gradient descent together with the network, uses a "softball" prior on the component means, and initialises new samples at the best component mean. This repository instead fits the GMM with EM through our own package [`tgmm`](https://adriansousapoza.github.io/tgmm/)
(several covariance types, see `model.gmm.covariance_type`), adds the latent noise, and initialises new samples at zero.
The principle (MAP over decoder, representations and mixture) is the same.

## 4. Repository structure

```text
ImageDGD/
├── config/
│   ├── config.yaml              # the base configuration: every hyperparameter lives here (Hydra / OmegaConf)
│   └── experiment/              # small override files, one per ablation (e.g. latent_dim_16.yaml)
├── src/
│   ├── data/dataloader.py       # FashionMNIST pooling, train/val/test split, label helpers
│   ├── models/
│   │   ├── representation_layer.py   # RepresentationLayer: the N x d table of z_i (init distributions, save/load)
│   │   ├── dgd.py                    # ConvDecoder (the image-specific part) and the DGD container
│   │   └── pca.py                    # optional PCA initialisation of Z
│   ├── training/trainer.py      # DGDTrainer: the training loop (Algorithm 1) and all training options
│   ├── utils/                   # device setup, noise schedules, checkpoint save/load
│   └── visualization/           # latent-space plots, reconstructions, noise diagnostics, animations, report.py
├── notebooks/                   # intro_to_dgd (this one), dgd_training_demo, dgd_test_inference,
│   │                            # run_experiments, compare_experiments, noise_injection_explained, ...
│   └── assets/intro/            # figures and videos used by this notebook
├── docs/                        # training_options.md (advanced options)
├── data/                        # FashionMNIST is downloaded here on first use      (git-ignored)
├── experiments/                 # one timestamped folder per run: config, checkpoints, figures  (git-ignored)
└── models/                      # ad-hoc saved models                                (git-ignored)
```

| Concept | Where to look |
|---|---|
| Representations $Z$ | `src/models/representation_layer.py` → `RepresentationLayer` |
| Decoder $f_\theta$ | `src/models/dgd.py` → `ConvDecoder` |
| GMM prior | `tgmm.GaussianMixture`, built in `DGDTrainer._create_model_components` from `model.gmm` |
| Training loop (Algorithm 1) | `src/training/trainer.py` → `DGDTrainer.train` |
| Inference (Algorithm 2) | `notebooks/dgd_test_inference.ipynb` |
| Noise schedules | `src/utils/schedules.py` |
| Checkpoints | `src/utils/checkpoint.py` (`save_checkpoint`, `load_checkpoint`) |
| Figures of a run | `src/visualization/report.py` → `generate_training_figures`, `generate_inference_figures` |

Each run writes to `experiments/<timestamp>_<experiment_name>/` with `config.yaml` (the exact settings used),
`models/best/` (decoder, representations, GMM, loss history), `models/checkpoints/` (every `plot_interval` epochs)
and `figures/`.

## 5. Running the code

**Install** (see the README for the RAPIDS/GPU variant): `uv venv .venv --python 3.12 && uv pip install --python .venv -r requirements.txt`.
FashionMNIST is downloaded automatically on the first run.

There are two ways to train, from simplest to most flexible:

1. **Notebook:** open `notebooks/dgd_training_demo.ipynb` and run all cells (about 10 minutes for 200 epochs on a GPU).
   It creates a timestamped run folder, trains, and writes all figures. Then run `dgd_test_inference.ipynb` on the held-out test set.
2. **Ablations:** put a few overrides in `config/experiment/<name>.yaml`, list the name in `run_experiments.ipynb`, and compare runs with `compare_experiments.ipynb`.

Both use the same training class, `DGDTrainer` in `src/training/trainer.py`.

## 6. The configuration file

Everything is controlled by `config/config.yaml`, loaded with [Hydra](https://hydra.cc)/OmegaConf. It has four top-level blocks:

```yaml
random_seed: 42
experiment_name: "ImageDGD_Default"   # name of the run folder
paths: ...                                  # where runs are written (set per run by the notebooks)
data:      ...                              # dataset, split, batch size
model:                                      # what the model is
  representation: ...                       #   size and initialisation of z
  decoder: ...                              #   ConvDecoder architecture
  gmm: ...                                  #   the mixture prior (passed to tgmm)
training:  ...                              # how it is trained: schedule, noise, optimisers
```

An **experiment file** in `config/experiment/` only lists what changes and is merged on top of the base config. For example `latent_dim_16.yaml`:

```yaml
# Experiment: latent dimension 16 instead of 8. latent_noise_start/end are
# already dimension-independent (a target displacement *norm* in z-space --
# noise_std_for_dim divides by sqrt(n_features) internally, see
# config.yaml's comment), so they do NOT need rescaling here; same for
# model.gmm.n_features (null -> inherits representation.n_features).
experiment_name: "ImageDGD_LatentDim16"
description: "Latent dimension raised from 8 to 16"

model:
  representation:
    n_features: 16
```

### Hyperparameters

Defaults as in `config/config.yaml`. The last column is a rule of thumb, not a guarantee. Ablating these is what the `config/experiment/` files are for.

**Data**

| Parameter | Default | Meaning |
|---|---|---|
| `data.batch_size` | 256 | images per decoder step; the decoder takes one step per batch, so more batches means more decoder steps per epoch |
| `data.val_split`, `data.test_split` | 0.1, 0.1 | fraction of the 70,000 pooled images used for validation / held-out test; the rest is training |
| `data.total_subset_fraction` | 1.0 | use only a fraction of the data (handy for quick tests) |

**Model**

| Parameter | Default | Meaning |
|---|---|---|
| `model.representation.n_features` | 4 | latent dimension $d$ of each $z_i$; larger = better reconstruction, harder to cluster/visualise |
| `model.representation.distribution` | `zeros` | how $Z$ is initialised (`zeros`, `normal`, `uniform_ball`, `pca`, ...) |
| `model.representation.n_latent_copies` | 1 | independent representations per image (experimental, see `config.yaml`) |
| `model.decoder.hidden_dims` | [64, 32] | channels of the conv blocks (high to low); sets decoder capacity |
| `model.decoder.init_size` | [7, 7] | spatial size right after the first Linear layer, upsampled to `output_size` |
| `model.decoder.activation`, `.normalization` | `leaky_relu`, `batch` | hidden activation and normalisation layers |
| `model.decoder.final_activation` | `sigmoid` | output activation; `sigmoid` because pixels are in [0, 1] |
| `model.decoder.upsampling_mode` | `nearest` | `nearest`/`bilinear` = Upsample + Conv, `transpose` = transposed convolution |
| `model.gmm.n_components` | 20 | number of mixture components $K$ (FashionMNIST has 10 classes; the GMM need not match that) |
| `model.gmm.covariance_type` | `tied_spherical` | shape of the components: `full`, `diag`, `spherical` and `tied_` variants |
| `model.gmm.reg_covar` | 1e-06 | floor added to the covariances; larger prevents collapse onto single points |

**Training**

| Parameter | Default | Meaning |
|---|---|---|
| `training.epochs` | 200 | passes over the training set |
| `training.first_epoch_gmm` | 50 | epoch where the GMM is first fitted and the prior loss switches on |
| `training.refit_gmm_interval` | 50 | epochs between full GMM refits (cheap warm-started updates in between) |
| `training.lambda_gmm` | 1.0 | weight $\lambda$ of the prior loss relative to reconstruction |
| `training.latent_noise_enabled` | true | add noise $\epsilon$ to $z$ during training |
| `training.latent_noise_start` → `_end` | 1.0 → 0.01 | expected size of the noise displacement in $z$-space, annealed over training |
| `training.latent_noise_schedule` | `cosine` | shape of the anneal: `constant`, `linear`, `cosine`, `warmup_cosine` |
| `training.optimizer.decoder.lr` / `.weight_decay` | 0.01 / 0.01 | AdamW for the decoder (stepped every batch) |
| `training.optimizer.representation.lr` / `.weight_decay` | 0.03 / 0.0 | AdamW for $Z$ (stepped once per epoch, hence the larger lr) |
| `training.lr_scheduler.*` | cosine, 0.01→0.001 (decoder), 0.03→0.003 ($Z$) | cosine decay of both learning rates |
| `training.early_stopping_patience` | `None` | stop if validation reconstruction has not improved for this many epochs after the GMM starts |
| `training.plot_interval` | 10 | checkpoint every N epochs for the progression plots/animations |
| `training.inference.epochs` | 400 | optimisation steps $M$ when representing new images |
| `training.inference.prior_warmup_steps` | `first_epoch_gmm` | steps $M_0$ with reconstruction only, before the prior is added |

More options (Muon optimiser, decoder EMA, growing GMM, cluster jumps, latent walk, loss reduction) are all **off by
default** and described in [`docs/training_options.md`](../docs/training_options.md).

## 7. Reading the results of a training run

All figures and videos in sections 7 and 8 come from one run with the default config (4-D latent, 20 components, 200 epochs, about 11 minutes on a GPU).
The figures are produced by `generate_training_figures`; the labels (the 10 clothing classes) are **never used in training**.
They only colour the plots and feed AMI/ARI, which measure how well the GMM clusters agree with the classes.

### Loss curves

<img src="assets/intro/loss_curves.png" alt="loss curves" width="100%">

* **Top row:** total loss and reconstruction loss (sum of squared pixel errors per image). Train and validation curves lie on top of each other: the validation images are represented as well as the training ones.
* At epoch 50 the **total loss jumps** because the prior term is added (bottom left, about +4). The reconstruction loss itself keeps falling.
* **Bottom right:** AMI/ARI of the GMM clusters vs. the 10 classes. In this run the validation AMI reaches about 0.49 and ARI about 0.28. Clusters are partly aligned with classes, but this is clearly *not* a classifier. Many classes (shirt, pullover, coat, T-shirt) overlap in a 4-D latent space.

### The latent space over training

<video src="assets/intro/train_latent_progression.mp4" controls autoplay loop muted playsinline width="100%">
Your viewer cannot play videos. Open <a href="assets/intro/train_latent_progression.mp4">train_latent_progression.mp4</a>.
</video>

Each frame is a checkpoint (every 10 epochs). Each point is one training image's $z_i$, projected to 2-D by PCA, UMAP and t-SNE, and coloured by class.
Before epoch 50 there is no GMM and the cloud is diffuse with only a rough class gradient. From epoch 50 the GMM means (×) and 95% ellipses appear, and classes start to separate (trousers, bags, footwear first).

The final latent space (epoch 200), and the 20 GMM components fitted on it (each colour is one component):

<img src="assets/intro/latent_final.png" alt="latent final" width="100%">

<img src="assets/intro/gmm_clusters.png" alt="gmm clusters" width="100%">

### What does one mixture component look like?

Component 2 (mixture weight 0.072) highlighted in the three projections. It occupies one compact region in each of them:

<img src="assets/intro/cluster02_latent.png" alt="component 2 in the latent space" width="100%">

Sampling $z$ from this component and decoding gives new images that were never in the training set:

<img src="assets/intro/cluster02_samples.png" alt="samples from component 2" width="100%">

All 32 samples are long-sleeved tops (pullovers/sweaters), varying in brightness and width: the component has captured one coherent kind of image. That is what the GMM prior buys: a generative model whose mixture components each describe a type of image.
Not every component is this clean. Component 1 (`figures/samples/component_rank01_samples.png` in the run folder), for example, mixes garment types, which is not surprising with 20 overlapping components in a 4-D latent space.

### Reconstructions

Validation images (ground truth, then reconstruction), obtained by optimising $z$ against the frozen decoder:

<img src="assets/intro/recon_val.png" alt="reconstructions" width="100%">

Reconstructions are smooth and blurry, which is typical for a small decoder, a squared-error loss and only 4 latent dimensions. The video shows training images being reconstructed at every checkpoint, from a generic blur to class-specific shapes:

<video src="assets/intro/train_recon_progression.mp4" controls autoplay loop muted playsinline width="100%">
Your viewer cannot play videos. Open <a href="assets/intro/train_recon_progression.mp4">train_recon_progression.mp4</a>.
</video>

## 8. Inference: representing new images

Because there is no encoder, a trained DGD has no direct way to turn a *new* image into a latent vector. Instead, we do for the new image what training did for the training images, but with everything else held fixed. This is **Algorithm 2** and is run by `notebooks/dgd_test_inference.ipynb` on the held-out **test set** (7000 images that neither the decoder nor the GMM has seen; the notebook re-derives the split from the same seed and checks that the data settings match the training run).

### What happens

* The decoder $\theta$ and the GMM are **frozen** (the decoder is in eval mode, so BatchNorm uses its running statistics and test images do not influence each other). Only the new representations $Z_{\text{test}}$ are optimised.
* Every test image starts at $z=0$ and gets $M$ = `training.inference.epochs` = 400 gradient steps with AdamW (learning rate decaying from 0.03 to 0.003).
* The same **noise injection** as in training is used, with its own schedule annealed over the $M$ steps.
* **Two phases.** For the first $M_0$ = `prior_warmup_steps` = 50 steps only the reconstruction loss is used. A fresh $z$ starts far from every mixture component, so adding the prior immediately would let its gradient dominate before $z$ has learned anything from the image. After step $M_0$ the prior term $-\lambda \log p_{\text{GMM}}(\tilde z)$ is added.
* The result is one $z^*$ per test image. Decoding it gives the **reconstruction**, and `GMM.predict(`$z^*$`)` gives a **cluster assignment**, which we compare to the true classes with AMI/ARI (the labels are not used in the optimisation).
* The price of having no encoder: representing 7,000 images took 400 optimisation steps, about 75 seconds on a laptop GPU, instead of one forward pass.

The run folder gets a new `inference/<timestamp>/` subfolder with the optimised test representations and all the figures below.

### Loss during inference

<img src="assets/intro/inference_loss_curve.png" alt="inference loss curves" width="100%">

* The reconstruction loss starts near 50 (a generic, blurry image, since $z=0$) and flattens out after about 350 steps.
* At step 50 the **prior switches on**: the GMM term jumps to about 7 and then falls to about 2.4 as the latents settle into mixture components, and the total loss jumps with it. This is the same jump you saw at epoch 50 of training.
* Bottom right: the noise schedule (cosine, over the inference steps).

> **A note on units.** The reconstruction term in this particular inference run was a smooth-L1 (Huber) loss, which for pixel values in [0, 1] equals ½ × the squared error that training uses. Multiply the reconstruction numbers in this plot by 2 to compare them with the training curves: the final 7.33 becomes 14.67, essentially the final validation reconstruction loss (14.67).

### The test latent space

<img src="assets/intro/test_latent_overview.png" alt="test latent space" width="100%">

Test AMI 0.487 and ARI 0.275, practically the same as the validation scores at the end of training (0.488 and 0.276). The test images land in the same mixture structure as the training images, so there is no sign of a generalisation gap in the clustering. The plot has fewer points than the training one (7,000 instead of 56,000), which is why the UMAP and t-SNE views look sparser.

The video shows how the 7,000 test latents move during the 400 steps. At the start they sit in a tight cloud around the origin; they spread out and, from step 50, are pulled towards the GMM components (×) while the noise shrinks:

<video src="assets/intro/inference_latent_progression.mp4" controls autoplay loop muted playsinline width="100%">
Your viewer cannot play videos. Open <a href="assets/intro/inference_latent_progression.mp4">inference_latent_progression.mp4</a>.
</video>

### Test reconstructions

<img src="assets/intro/test_recon.png" alt="test reconstructions" width="100%">

Five test images per class (ground truth, then the reconstruction decoded from the optimised $z^*$). Simple, frequent shapes (trousers, sneakers, ankle boots) are reproduced well; fine detail such as prints, straps and sandal structure is lost, as expected from a 4-D latent. The video shows the same images being reconstructed as the optimisation proceeds:

<video src="assets/intro/inference_recon_progression.mp4" controls autoplay loop muted playsinline width="100%">
Your viewer cannot play videos. Open <a href="assets/intro/inference_recon_progression.mp4">inference_recon_progression.mp4</a>.
</video>

## 9. Where to go next

**Notebooks**

| Notebook | What it does |
|---|---|
| `dgd_training_demo.ipynb` | full training run with figures |
| `dgd_test_inference.ipynb` | Algorithm 2: represent the held-out test set with the frozen model |
| `noise_injection_explained.ipynb` | what latent noise does and why it is annealed |
| `representation_layer_demo.ipynb` | the initial distributions of $Z$ |
| `run_experiments.ipynb`, `compare_experiments.ipynb` | batch of ablations from `config/experiment/`, and a comparison of runs |

**Exercises**

1. Train with `model.representation.n_features` set to 2, 4 (the default) and 16 (`config/experiment/latent_dim_16.yaml`; write your own override file for 2). How do the reconstructions and the latent plots change?
2. Set `training.latent_noise_enabled: false`. What happens to the latent space? (compare with `no_noise_baseline.yaml`)
3. Vary `model.gmm.n_components` (15, 20, 25) and `covariance_type`. Does AMI follow?
4. Move `first_epoch_gmm` to 25 or 100. How does the epoch-`first_epoch_gmm` jump in the loss curve change?
5. Open `DGDTrainer.train` and find the three places where the steps of section 3 are implemented: the training batch loop, the validation loop and the GMM refit.
6. Run `dgd_test_inference.ipynb` with `training.inference.epochs` set to 100 and to 800. How do the reconstruction and the test AMI change, and how long does it take?

## References

Papers from the Krogh group, in the order they are useful here:

1. Schuster, V. & Krogh, A. (2021). *A Manifold Learning Perspective on Representation Learning: Learning Decoder and Representations without an Encoder.* Entropy 23(11), 1403. doi:10.3390/e23111403
2. Schuster, V. & Krogh, A. (2023). *The Deep Generative Decoder: MAP estimation of representations improves modelling of single-cell RNA data.* Bioinformatics 39(9), btad497. doi:10.1093/bioinformatics/btad497
3. Schuster, V., Dann, E., Krogh, A. & Teichmann, S. A. (2024). *multiDGD: A versatile deep generative model for multi-omics data.* Nature Communications 15, 10031. doi:10.1038/s41467-024-53340-z
4. Prada-Luengo, I., Schuster, V., Liang, Y., Terkelsen, T., Sora, V. & Krogh, A. (2023). *N-of-one differential gene expression without control samples using a deep generative model.* Genome Biology 24, 263. doi:10.1186/s13059-023-03104-7
5. Bjerregaard, A., Prada-Luengo, I., Das, V. & Krogh, A. (2025). *What Do Single-Cell Models Already Know About Perturbations?* Genes 16(12), 1439. doi:10.3390/genes16121439

GMM package: [`tgmm`](https://adriansousapoza.github.io/tgmm/).